# M4a — Extração de texto do conjunto de teste holdout

Roda a MESMA extração local (PyMuPDF nativo + PaddleOCR) do notebook 02, mas sobre `data/documento_social/teste_holdout/arquivos/` — documentos **fora** do treino e da validação usados nos notebooks 01-06, para testar o modelo em dados genuinamente novos.

**Como usar:** coloque os PDFs/imagens que você quer testar em `data/documento_social/teste_holdout/arquivos/`, rode este notebook até o fim. Em seguida (em qualquer ordem, já que ambos partem só dos arquivos originais): rode `notebooks/07.gerar_ground_truth_holdout.ipynb` (se ainda não tiver gabarito para esses documentos) e depois `notebooks/09.alinhar_ground_truth_holdout.ipynb`, que casa o gabarito do 07 com a extração deste notebook — necessário para o notebook 10 (métricas). Se só quiser inspecionar visualmente as previsões sem gabarito, pode pular direto para `notebooks/11.verificar_resultados_manualmente.ipynb`.

**⚠️ Mesma restrição de ambiente do notebook 02: use o kernel `locscan-ocr` (`.venv_ocr`), NUNCA o venv geral do projeto** — paddlepaddle/paddleocr não pode coexistir no mesmo processo Python com torch/transformers no Windows (conflito de cuDNN).

In [1]:
import os
os.environ.setdefault("FLAGS_use_mkldnn", "0")

import json
from pathlib import Path

import cv2
import numpy as np
import pymupdf

## Configuração de caminhos

In [2]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()

PASTA_ENTRADA = RAIZ / "data" / "documento_social" / "teste_holdout" / "arquivos"
PASTA_SAIDA = RAIZ / "data" / "documento_social" / "teste_holdout" / "extracao"
PASTA_ENTRADA.mkdir(parents=True, exist_ok=True)
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

DPI_RENDER = 300
MIN_CHARS_TEXTO_NATIVO = 20
MIN_SPREAD_VERTICAL_TEXTO_NATIVO = 0.30
MIN_RATIO_LEGIVEL_TEXTO_NATIVO = 0.70
SOBRESCREVER = False

EXTENSOES_PDF = {".pdf"}
EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

print(f"Raiz do projeto: {RAIZ}")
print(f"Entrada (holdout): {PASTA_ENTRADA}")
print(f"Saída:             {PASTA_SAIDA}")
print(f"\nColoque os documentos de teste em {PASTA_ENTRADA} antes de continuar.")

Raiz do projeto: d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3
Entrada (holdout): d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\arquivos
Saída:             d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\extracao

Coloque os documentos de teste em d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\data\documento_social\teste_holdout\arquivos antes de continuar.


## Pré-processamento de imagem (OpenCV) — igual ao notebook 02

In [3]:
def corrigir_inclinacao(img_cinza: np.ndarray) -> np.ndarray:
    """Detecta o ângulo dominante do texto via Hough e rotaciona para nivelar a página."""
    bordas = cv2.Canny(img_cinza, 50, 150, apertureSize=3)
    linhas = cv2.HoughLinesP(bordas, 1, np.pi / 180, threshold=200, minLineLength=img_cinza.shape[1] // 4, maxLineGap=20)

    if linhas is None or len(linhas) == 0:
        return img_cinza

    angulos = []
    for linha in linhas:
        x1, y1, x2, y2 = linha[0]
        angulo = np.degrees(np.arctan2(y2 - y1, x2 - x1))
        if -45 < angulo < 45:
            angulos.append(angulo)

    if not angulos:
        return img_cinza

    angulo_mediano = float(np.median(angulos))
    if abs(angulo_mediano) < 0.3:
        return img_cinza

    h, w = img_cinza.shape[:2]
    centro = (w // 2, h // 2)
    matriz = cv2.getRotationMatrix2D(centro, angulo_mediano, 1.0)
    return cv2.warpAffine(img_cinza, matriz, (w, h), flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def preprocessar_imagem(img_bgr: np.ndarray) -> np.ndarray:
    cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    cinza = corrigir_inclinacao(cinza)
    cinza = cv2.fastNlMeansDenoising(cinza, h=7, templateWindowSize=7, searchWindowSize=21)
    binarizada = cv2.adaptiveThreshold(
        cinza, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, blockSize=31, C=15
    )
    return binarizada

## Motor OCR (PaddleOCR)

In [4]:
import paddle
from paddleocr import PaddleOCR

cuda_disponivel = paddle.device.is_compiled_with_cuda()
n_gpus = paddle.device.cuda.device_count() if cuda_disponivel else 0
print(f"Paddle compilado com CUDA: {cuda_disponivel} | GPUs detectadas: {n_gpus}")

OCR = PaddleOCR(
    device="gpu:0",
    lang="pt",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    return_word_box=True,
    enable_mkldnn=False,
)

d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\.venv_ocr\lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:717: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
d:\Arquivos\ProjetosPython\BFC\LocalScanDocument_v3\.venv_ocr\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating model: ('PP-OCRv6_medium_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Gabriel-PC\.paddlex\official_models\PP-OCRv6_medium_det`.


Paddle compilado com CUDA: True | GPUs detectadas: 1


Creating model: ('PP-OCRv6_medium_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\Gabriel-PC\.paddlex\official_models\PP-OCRv6_medium_rec`.


## Extração por página: nativa ou OCR (com correção de tokens fragmentados — ver anotacoes.txt 2026-09-11)

In [ ]:
def extrair_pagina_nativa(pagina: "pymupdf.Page") -> list:
    palavras = pagina.get_text("words")
    tokens = []
    for x0, y0, x1, y1, texto, *_ in palavras:
        texto = texto.strip()
        if texto:
            tokens.append({"text": texto, "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)]})
    return tokens


def _box_para_bbox(box) -> list:
    box = np.asarray(box, dtype=float)
    if box.ndim == 1:
        x0, y0, x1, y1 = box[:4]
        return [float(x0), float(y0), float(x1), float(y1)]
    xs = box[:, 0]
    ys = box[:, 1]
    return [float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())]


def mesclar_tokens_fragmentados(tokens: list) -> list:
    """Junta tokens que o PaddleOCR partiu incorretamente no meio de uma palavra (ex.: acento
    isolado). Ver notebooks/02.extrair_texto_ocr_local.ipynb para a versão canônica/comentada."""
    if not tokens:
        return tokens
    tokens_ordenados = sorted(tokens, key=lambda t: (round(t["bbox"][1], 0), t["bbox"][0]))
    mesclados = [dict(tokens_ordenados[0])]
    for tok in tokens_ordenados[1:]:
        anterior = mesclados[-1]
        altura_ref = max(anterior["bbox"][3] - anterior["bbox"][1], 1.0)
        mesma_linha = abs(tok["bbox"][1] - anterior["bbox"][1]) < altura_ref * 0.5
        gap = tok["bbox"][0] - anterior["bbox"][2]
        fragmento_isolado = len(tok["text"]) <= 2 and not tok["text"][:1].isupper()
        if mesma_linha and fragmento_isolado and 0 <= gap < altura_ref * 0.35:
            anterior["text"] = anterior["text"] + tok["text"]
            anterior["bbox"][2] = tok["bbox"][2]
            anterior["bbox"][1] = min(anterior["bbox"][1], tok["bbox"][1])
            anterior["bbox"][3] = max(anterior["bbox"][3], tok["bbox"][3])
        else:
            mesclados.append(dict(tok))
    return mesclados


def _dpi_efetivo_pagina(pagina, dpi_padrao: int = DPI_RENDER, cobertura_minima: float = 0.75) -> int:
    """Calcula o DPI que reproduz a resolução NATIVA da imagem escaneada dominante da página, em
    vez de sempre usar `dpi_padrao` fixo. Achado 2026-09-21: alguns PDFs escaneados declaram um
    MediaBox bem maior que o conteúdo real (ex.: scan A4 dentro de um MediaBox de ~21x30
    polegadas -- "Constituição 12-09-1996.pdf" do holdout). Nesses casos, `get_pixmap(dpi=300)`
    faz o PyMuPDF esticar/interpolar a imagem original (resolução fixa) sobre uma área bem maior
    que a real, produzindo uma imagem "borrada" com MENOS densidade de informação por área do que
    o DPI nominal sugere -- e o OCR sai pior nessas páginas. Usar o DPI calculado a partir da
    proporção real (pixels nativos da imagem / tamanho do bbox onde ela está desenhada) evita
    essa interpolação: quando o resultado é MENOR que `dpi_padrao`, ainda assim reproduz a imagem
    de forma mais fiel (praticamente 1:1, sem esticar) do que insistir no DPI fixo sobre uma
    página maior que o conteúdo real."""
    try:
        # Regressão real encontrada 2026-09-21 (documento "Alteração Consolidada 102019.pdf",
        # páginas 2/5/6/7): páginas com uma imagem de fundo quase em branco (papel de segurança/
        # marca d'água) cobrindo quase toda a área, mas com o conteúdo de verdade em TEXTO NATIVO
        # por cima -- calcular um DPI baseado só na imagem, nesse caso, é inofensivo aqui (esta
        # função só decide o DPI do get_pixmap, que sempre compõe texto+imagem corretamente), mas
        # ainda assim não faz sentido recalcular DPI a partir de uma imagem de fundo decorativa
        # quando a página tem texto nativo de verdade -- usa o DPI padrão nesse caso.
        if len(pagina.get_text().strip()) >= 20:
            return dpi_padrao
        area_pagina = pagina.rect.width * pagina.rect.height
        if area_pagina <= 0:
            return dpi_padrao
        melhor_bbox, melhor_area, melhor_largura_px = None, 0, None
        for info in pagina.get_image_info(xrefs=True):
            bbox = info.get("bbox")
            if not bbox:
                continue
            area_imagem = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            if area_imagem > melhor_area:
                melhor_area, melhor_bbox, melhor_largura_px = area_imagem, bbox, info.get("width")
        if melhor_bbox is None or not melhor_largura_px or (melhor_area / area_pagina) < cobertura_minima:
            return dpi_padrao
        largura_bbox_pt = melhor_bbox[2] - melhor_bbox[0]
        if largura_bbox_pt <= 0:
            return dpi_padrao
        return max(72, round((melhor_largura_px / largura_bbox_pt) * 72))
    except Exception:
        return dpi_padrao


def extrair_pagina_ocr(pagina: "pymupdf.Page", dpi: int = DPI_RENDER) -> list:
    dpi = _dpi_efetivo_pagina(pagina, dpi)
    pix = pagina.get_pixmap(dpi=dpi)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR if pix.n == 4 else cv2.COLOR_RGB2BGR)

    img_processada = preprocessar_imagem(img_bgr)
    img_ocr = cv2.cvtColor(img_processada, cv2.COLOR_GRAY2BGR)

    resultado = OCR.predict(img_ocr)
    if not resultado:
        return []
    r = resultado[0]

    escala = 72.0 / dpi
    tokens = []
    for palavras_linha, boxes_linha in zip(r.get("text_word", []), r.get("text_word_boxes", [])):
        tokens_linha = []
        for palavra, box in zip(palavras_linha, boxes_linha):
            palavra = palavra.strip()
            if not palavra:
                continue
            x0, y0, x1, y1 = _box_para_bbox(box)
            tokens_linha.append({
                "text": palavra,
                "bbox": [round(x0 * escala, 2), round(y0 * escala, 2), round(x1 * escala, 2), round(y1 * escala, 2)],
            })
        tokens.extend(mesclar_tokens_fragmentados(tokens_linha))

    return tokens

## Triagem e processamento de um documento completo

In [6]:
def _ratio_caracteres_legiveis(texto: str) -> float:
    if not texto:
        return 0.0
    validos = sum(1 for c in texto if c.isalpha() or c.isdigit() or c.isspace() or c in ".,;:-/()ºª%")
    return validos / len(texto)


def tem_texto_nativo(pagina: "pymupdf.Page") -> bool:
    texto = pagina.get_text().strip()
    if len(texto) < MIN_CHARS_TEXTO_NATIVO:
        return False
    if _ratio_caracteres_legiveis(texto) < MIN_RATIO_LEGIVEL_TEXTO_NATIVO:
        return False
    palavras = pagina.get_text("words")
    if not palavras:
        return False
    altura_pagina = pagina.rect.height
    if altura_pagina <= 0:
        return True
    ys = [w[1] for w in palavras]
    spread_relativo = (max(ys) - min(ys)) / altura_pagina
    return spread_relativo >= MIN_SPREAD_VERTICAL_TEXTO_NATIVO


def processar_pdf(caminho: Path) -> dict:
    doc = pymupdf.open(str(caminho))
    paginas_saida = []
    for i, pagina in enumerate(doc, start=1):
        if tem_texto_nativo(pagina):
            tokens = extrair_pagina_nativa(pagina)
            metodo = "nativo"
        else:
            tokens = extrair_pagina_ocr(pagina)
            metodo = "ocr"
        paginas_saida.append({"page": i, "method": metodo, "tokens": tokens})
    doc.close()
    return {"document_id": caminho.stem, "source_file": caminho.name, "pages": paginas_saida}


def processar_imagem(caminho: Path) -> dict:
    img_bgr = cv2.imread(str(caminho))
    img_processada = preprocessar_imagem(img_bgr)
    img_ocr = cv2.cvtColor(img_processada, cv2.COLOR_GRAY2BGR)

    resultado = OCR.predict(img_ocr)
    tokens = []
    if resultado:
        r = resultado[0]
        for palavras_linha, boxes_linha in zip(r.get("text_word", []), r.get("text_word_boxes", [])):
            tokens_linha = []
            for palavra, box in zip(palavras_linha, boxes_linha):
                palavra = palavra.strip()
                if not palavra:
                    continue
                x0, y0, x1, y1 = _box_para_bbox(box)
                tokens_linha.append({"text": palavra, "bbox": [round(x0, 2), round(y0, 2), round(x1, 2), round(y1, 2)]})
            tokens.extend(mesclar_tokens_fragmentados(tokens_linha))

    return {"document_id": caminho.stem, "source_file": caminho.name, "pages": [{"page": 1, "method": "ocr", "tokens": tokens}]}

## Loop principal

In [7]:
def listar_arquivos_suportados(pasta: Path) -> list:
    if not pasta.is_dir():
        return []
    return [p for p in pasta.iterdir() if p.is_file() and p.suffix.lower() in (EXTENSOES_PDF | EXTENSOES_IMAGEM)]


arquivos = sorted(listar_arquivos_suportados(PASTA_ENTRADA))

if not arquivos:
    raise SystemExit(f"Nenhum arquivo suportado encontrado em {PASTA_ENTRADA} -- coloque PDFs/imagens lá antes de rodar.")

print(f"Encontrados {len(arquivos)} arquivo(s) em {PASTA_ENTRADA.name}")

for i, caminho in enumerate(arquivos, start=1):
    destino = PASTA_SAIDA / f"{caminho.stem}.json"
    if destino.exists() and not SOBRESCREVER:
        print(f"[{i}/{len(arquivos)}] {caminho.name}: já existe, pulando (SOBRESCREVER=True para refazer)")
        continue

    print(f"[{i}/{len(arquivos)}] {caminho.name}: processando...")
    try:
        if caminho.suffix.lower() in EXTENSOES_PDF:
            registro = processar_pdf(caminho)
        else:
            registro = processar_imagem(caminho)
    except Exception as e:
        print(f"[{i}/{len(arquivos)}] {caminho.name}: ERRO -> {e}")
        continue

    destino.write_text(json.dumps(registro, ensure_ascii=False, indent=2), encoding="utf-8")

    n_tokens = sum(len(p["tokens"]) for p in registro["pages"])
    metodos = {p["page"]: p["method"] for p in registro["pages"]}
    print(f"[{i}/{len(arquivos)}] {caminho.name}: {len(registro['pages'])} página(s), {n_tokens} token(s) | métodos: {metodos} -> {destino.name}")

print("\nConcluído. Próximo passo: notebooks/07.gerar_ground_truth_holdout.ipynb + notebooks/09.alinhar_ground_truth_holdout.ipynb (kernel geral, .venv) para gerar métricas, ou notebooks/11.verificar_resultados_manualmente.ipynb para só inspecionar visualmente.")

Encontrados 30 arquivo(s) em arquivos
[1/30] cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf.pdf: processando...
[1/30] cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf.pdf: 10 página(s), 3002 token(s) | métodos: {1: 'nativo', 2: 'nativo', 3: 'nativo', 4: 'nativo', 5: 'nativo', 6: 'nativo', 7: 'nativo', 8: 'nativo', 9: 'nativo', 10: 'nativo'} -> cm1iasv3k0031dhqdfn4663iw-Alteração 11 - MINAS CAL.pdf.json
[2/30] cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.pdf: processando...
[2/30] cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.pdf: 3 página(s), 1366 token(s) | métodos: {1: 'ocr', 2: 'ocr', 3: 'ocr'} -> cm1jln7kb006wdiqdck8ievgk-Constituição 12-09-1996.pdf.json
[3/30] cm1ow1j3e0008uiqd53mf55j4-CONTRATO SOCIAL NOVO 22.03.pdf.pdf: processando...
[3/30] cm1ow1j3e0008uiqd53mf55j4-CONTRATO SOCIAL NOVO 22.03.pdf.pdf: 6 página(s), 1558 token(s) | métodos: {1: 'ocr', 2: 'ocr', 3: 'ocr', 4: 'ocr', 5: 'ocr', 6: 'ocr'} -> cm1ow1j3e0008uiqd53mf55j4-CONTRATO SOCIAL NOVO

Resized image size (4819x6848) exceeds max_side_limit of 4000. Resizing to fit within limit.


[9/30] cm3zxn5i1007k6lqdfgssemj9-BRUNO MEDEIROS20012015_0017.jpg.jpg: 1 página(s), 172 token(s) | métodos: {1: 'ocr'} -> cm3zxn5i1007k6lqdfgssemj9-BRUNO MEDEIROS20012015_0017.jpg.json
[10/30] cm6dqdqda00c5t5qdbn57eym4-CONTRATO SOCIAL CONSTITUTIVO.pdf.pdf: processando...
[10/30] cm6dqdqda00c5t5qdbn57eym4-CONTRATO SOCIAL CONSTITUTIVO.pdf.pdf: 6 página(s), 1400 token(s) | métodos: {1: 'ocr', 2: 'ocr', 3: 'ocr', 4: 'ocr', 5: 'ocr', 6: 'nativo'} -> cm6dqdqda00c5t5qdbn57eym4-CONTRATO SOCIAL CONSTITUTIVO.pdf.json
[11/30] cm79dwyah00t7s4qda3jqdnck-PROCESSO_250219883_722025_181223 (3).pdf.pdf: processando...
[11/30] cm79dwyah00t7s4qda3jqdnck-PROCESSO_250219883_722025_181223 (3).pdf.pdf: 9 página(s), 1914 token(s) | métodos: {1: 'nativo', 2: 'nativo', 3: 'nativo', 4: 'nativo', 5: 'nativo', 6: 'nativo', 7: 'nativo', 8: 'nativo', 9: 'nativo'} -> cm79dwyah00t7s4qda3jqdnck-PROCESSO_250219883_722025_181223 (3).pdf.json
[12/30] cm7awuw5u000oxqqdg28s8vcq-Contrato Social.pdf.pdf: processando...
[12/30] 

Resized image size (2463x4238) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2467x4246) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2467x4246) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2467x4242) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2467x4242) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2467x4259) exceeds max_side_limit of 4000. Resizing to fit within limit.
Resized image size (2463x4221) exceeds max_side_limit of 4000. Resizing to fit within limit.


[27/30] cmoa66iv400mh01o7ao4u970q-contratosocial1pdf.pdf: 7 página(s), 2242 token(s) | métodos: {1: 'ocr', 2: 'ocr', 3: 'ocr', 4: 'ocr', 5: 'ocr', 6: 'ocr', 7: 'ocr'} -> cmoa66iv400mh01o7ao4u970q-contratosocial1pdf.json
[28/30] cmod12h77007m01nwb5eldybv-estatutoconsolidado2015pdf.pdf: processando...
[28/30] cmod12h77007m01nwb5eldybv-estatutoconsolidado2015pdf.pdf: 12 página(s), 4593 token(s) | métodos: {1: 'ocr', 2: 'ocr', 3: 'ocr', 4: 'ocr', 5: 'ocr', 6: 'ocr', 7: 'ocr', 8: 'ocr', 9: 'ocr', 10: 'ocr', 11: 'ocr', 12: 'ocr'} -> cmod12h77007m01nwb5eldybv-estatutoconsolidado2015pdf.json
[29/30] cmod1ex56006x01qu4epr6s43-2alteraocontratosocialpdf.pdf: processando...
[29/30] cmod1ex56006x01qu4epr6s43-2alteraocontratosocialpdf.pdf: 2 página(s), 537 token(s) | métodos: {1: 'ocr', 2: 'ocr'} -> cmod1ex56006x01qu4epr6s43-2alteraocontratosocialpdf.json
[30/30] cmsj91fmo005501qr6ino9miv-altconteconsolidacao081pdf.pdf: processando...
[30/30] cmsj91fmo005501qr6ino9miv-altconteconsolidacao081pdf.pdf: